# Crypto VC Investment Intelligence Report

**Period:** November 2024 - April 2025  
**Data Source:** Messari / CryptoRank  
**Author:** Phuc Le

This notebook analyzes venture capital investment activity in the blockchain and cryptocurrency sector. The analysis covers 338 unique funding rounds across 326 projects, involving 1,298 unique investors, with $6.9B in total capital deployed.

## Objectives

1. Identify the most active VCs and their investment patterns
2. Analyze capital flow across categories and funding stages
3. Map co-investment relationships and syndicate patterns
4. Provide strategic insights for infrastructure startups seeking funding


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import combinations
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Consistent color theme
COLORS = {
    'primary': '#1B3A5C', 'secondary': '#2E6B9E', 'accent': '#4A90D9',
    'light': '#A8C8E8', 'lightest': '#D6E6F5', 'gray_dark': '#4A4A4A',
    'gray_mid': '#8C8C8C', 'gray_light': '#D9D9D9', 'background': '#F5F7FA',
    'white': '#FFFFFF', 'highlight': '#E8913A',
}
BLUE_PALETTE = ['#1B3A5C','#2E6B9E','#3A7FC1','#4A90D9','#6BAAE0','#8BBDE6','#A8C8E8','#C5D9EF','#D6E6F5','#E8F0F8']

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11, 'axes.titlesize': 14,
    'axes.titleweight': 'bold', 'axes.facecolor': '#FFFFFF',
    'figure.facecolor': '#F5F7FA', 'axes.edgecolor': '#D9D9D9',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.color': '#D9D9D9',
    'text.color': '#4A4A4A',
})


## 2. Data Loading & Cleaning

The raw dataset was collected from Messari and CryptoRank. Each row represents an investor-deal pair (one row per investor per funding round). Key cleaning steps:
- Standardized 60+ raw category labels into 32 consistent categories
- Converted fundraise and pre-valuation from string to numeric (removing comma separators)
- Dropped 15 rows with missing investor names (data artifacts)
- Added derived columns: Month, Quarter, Valuation Multiple


In [ ]:
df = pd.read_csv('../data/vcs_investments_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Create unique deals view (one row per funding round)
unique_deals = df.drop_duplicates(subset=['Project', 'Date'])[
    ['Project','Date','Fundraise','Pre-valuation','Stage','Category','Month','Quarter','Year']
].copy()

print(f"Dataset: {len(df):,} investor-deal records")
print(f"Unique funding rounds: {len(unique_deals)}")
print(f"Unique projects: {df['Project'].nunique()}")
print(f"Unique investors: {df['Investors'].nunique()}")
print(f"Total capital deployed: ${unique_deals['Fundraise'].sum():,.0f}M")
print(f"Date range: {df['Date'].min().strftime('%b %Y')} - {df['Date'].max().strftime('%b %Y')}")
print(f"\nCategories: {df['Category'].nunique()}")
print(f"Stages: {df['Stage'].nunique()}")


## 3. Market Overview

### 3.1 Monthly Investment Activity


In [ ]:
monthly = unique_deals.groupby('Month').agg(
    deals=('Project','count'), capital=('Fundraise','sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(range(len(monthly)), monthly['capital'], color=COLORS['light'],
        edgecolor=COLORS['secondary'], linewidth=0.5, label='Capital ($M)', zorder=2)
ax1.set_ylabel('Capital Deployed ($M)', color=COLORS['secondary'])
ax2 = ax1.twinx()
ax2.plot(range(len(monthly)), monthly['deals'], color=COLORS['highlight'],
         marker='o', linewidth=2.5, markersize=8, label='Deal Count', zorder=3)
ax2.set_ylabel('Number of Deals', color=COLORS['highlight'])
ax1.set_xticks(range(len(monthly)))
ax1.set_xticklabels(monthly['Month'], rotation=45, ha='right')
ax1.set_title('Monthly Investment Activity: Capital Deployed vs Deal Count', pad=15)
ax1.spines['top'].set_visible(False); ax2.spines['top'].set_visible(False)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../charts/03_monthly_trend.png', dpi=200, bbox_inches='tight')
plt.show()


### 3.2 Capital by Category

In [ ]:
cat_invest = unique_deals.groupby('Category')['Fundraise'].sum().sort_values(ascending=True)
total = cat_invest.sum()

fig, ax = plt.subplots(figsize=(14, 10))
colors_gradient = [BLUE_PALETTE[min(int(i*len(BLUE_PALETTE)/len(cat_invest)), len(BLUE_PALETTE)-1)]
                   for i in range(len(cat_invest))]
bars = ax.barh(range(len(cat_invest)), cat_invest.values, color=colors_gradient, edgecolor='white')
ax.set_yticks(range(len(cat_invest)))
ax.set_yticklabels(cat_invest.index, fontsize=10)
ax.set_xlabel('Total Investment ($M)')
ax.set_title('Total Capital Deployed by Category', pad=15)
for i, (v, bar) in enumerate(zip(cat_invest.values, bars)):
    pct = v/total*100
    label = f'${v:.0f}M ({pct:.1f}%)' if pct >= 1 else f'${v:.0f}M'
    ax.text(v + total*0.01, bar.get_y()+bar.get_height()/2, label, va='center', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/02_investment_by_category.png', dpi=200, bbox_inches='tight')
plt.show()


### 3.3 Deal Size Distribution & Stage Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].hist(unique_deals['Fundraise'], bins=30, color=COLORS['accent'], edgecolor='white', alpha=0.9)
axes[0].axvline(unique_deals['Fundraise'].median(), color=COLORS['highlight'], linestyle='--',
                linewidth=2, label=f'Median: ${unique_deals["Fundraise"].median():.0f}M')
axes[0].set_xlabel('Fundraise Amount ($M)'); axes[0].set_ylabel('Number of Deals')
axes[0].set_title('Deal Size Distribution'); axes[0].legend()
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

stage_order = ['Pre-seed','Seed Round','Series A','Strategic Round','Funding Round','Private Sale']
stage_data = unique_deals[unique_deals['Stage'].isin(stage_order)]
bp = axes[1].boxplot([stage_data[stage_data['Stage']==s]['Fundraise'] for s in stage_order
                      if s in stage_data['Stage'].values],
                     labels=[s.replace(' Round','').replace(' Sale','') for s in stage_order
                             if s in stage_data['Stage'].values],
                     patch_artist=True, showfliers=True)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor(BLUE_PALETTE[i % len(BLUE_PALETTE)])
for median in bp['medians']: median.set_color(COLORS['highlight']); median.set_linewidth(2)
axes[1].set_ylabel('Fundraise Amount ($M)'); axes[1].set_title('Deal Size by Stage')
axes[1].tick_params(axis='x', rotation=30)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/04_deal_size_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Median deal size: ${unique_deals['Fundraise'].median():.0f}M")
print(f"Mean deal size: ${unique_deals['Fundraise'].mean():.1f}M")
print(f"\nStage breakdown:")
for stage in stage_order:
    sd = unique_deals[unique_deals['Stage']==stage]
    if len(sd)>0:
        print(f"  {stage}: {len(sd)} deals, median ${sd['Fundraise'].median():.0f}M")


## 4. VC Intelligence

### 4.1 Top 30 Most Active VCs


In [ ]:
vc_counts = df['Investors'].value_counts().head(30).reset_index()
vc_counts.columns = ['VC Firm', 'Investments']

fig, ax = plt.subplots(figsize=(14, 9))
bars = ax.barh(range(len(vc_counts)-1,-1,-1), vc_counts['Investments'],
               color=[BLUE_PALETTE[min(i//3, len(BLUE_PALETTE)-1)] for i in range(len(vc_counts))],
               edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(vc_counts)-1,-1,-1))
ax.set_yticklabels(vc_counts['VC Firm'], fontsize=10)
ax.set_xlabel('Number of Investments')
ax.set_title('Top 30 Most Active VCs in Crypto (Nov 2024 - Apr 2025)', pad=15)
for v, bar in zip(vc_counts['Investments'], bars):
    ax.text(v+0.3, bar.get_y()+bar.get_height()/2, str(v), va='center', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/01_top_30_active_vcs.png', dpi=200, bbox_inches='tight')
plt.show()


### 4.2 Co-Investment Heatmap

In [ ]:
top_20_vcs = df['Investors'].value_counts().head(20).index.tolist()
matrix = pd.DataFrame(0, index=top_20_vcs, columns=top_20_vcs)

for project, group in df.groupby('Project'):
    investors = [inv for inv in group['Investors'].unique() if inv in top_20_vcs]
    if len(investors) >= 2:
        for i in range(len(investors)):
            for j in range(i+1, len(investors)):
                matrix.loc[investors[i], investors[j]] += 1
                matrix.loc[investors[j], investors[i]] += 1
np.fill_diagonal(matrix.values, 0)

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(matrix, dtype=bool), k=0)
sns.heatmap(matrix, mask=mask, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='white', square=True, ax=ax, annot_kws={'size': 9})
ax.set_title('Co-Investment Frequency Among Top 20 VCs', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9); plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('../charts/07_coinvestment_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()


### 4.3 VC Concentration Analysis

In [ ]:
vc_deal_counts = df['Investors'].value_counts()
sorted_counts = vc_deal_counts.sort_values(ascending=False)
cumulative = sorted_counts.cumsum() / sorted_counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].fill_between(range(1, len(cumulative)+1), cumulative.values, color=COLORS['lightest'], alpha=0.8)
axes[0].plot(range(1, len(cumulative)+1), cumulative.values, color=COLORS['primary'], linewidth=2)
idx_50 = np.argmax(cumulative.values >= 50) + 1
idx_80 = np.argmax(cumulative.values >= 80) + 1
axes[0].axhline(50, color=COLORS['highlight'], linestyle='--', alpha=0.7)
axes[0].axhline(80, color=COLORS['highlight'], linestyle='--', alpha=0.7)
axes[0].annotate(f'Top {idx_50} VCs = 50%', xy=(idx_50, 50), fontsize=9, color=COLORS['highlight'])
axes[0].annotate(f'Top {idx_80} VCs = 80%', xy=(idx_80, 80), fontsize=9, color=COLORS['highlight'])
axes[0].set_xlabel('Number of VCs (ranked)'); axes[0].set_ylabel('Cumulative % of Investments')
axes[0].set_title('VC Concentration Curve')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

bins_labels = ['1','2','3-4','5-9','10-19','20+']
vc_binned = pd.cut(vc_deal_counts, bins=[0,1,2,4,9,19,50], labels=bins_labels)
bin_counts = vc_binned.value_counts().reindex(bins_labels)
axes[1].bar(range(len(bin_counts)), bin_counts.values, color=BLUE_PALETTE[:len(bin_counts)], edgecolor='white')
axes[1].set_xticks(range(len(bin_counts))); axes[1].set_xticklabels(bins_labels)
axes[1].set_xlabel('Number of Investments'); axes[1].set_ylabel('Number of VC Firms')
axes[1].set_title('VC Activity Distribution')
for i, v in enumerate(bin_counts.values):
    if not np.isnan(v): axes[1].text(i, v+5, str(int(v)), ha='center', fontsize=10)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/11_vc_concentration.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Total active investors: {len(vc_deal_counts):,}")
print(f"Top {idx_50} VCs account for 50% of all investments")
print(f"Top {idx_80} VCs account for 80% of all investments")


## 5. Infrastructure Deep Dive

This section focuses specifically on Infrastructure investments, providing actionable intelligence for startups in this category.

### 5.1 Top VCs in Infrastructure


In [ ]:
infra_df = df[df['Category'] == 'Infrastructure']
infra_deals = unique_deals[unique_deals['Category'] == 'Infrastructure']

infra_vc_counts = infra_df['Investors'].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

bars = axes[0].barh(range(len(infra_vc_counts)-1,-1,-1), infra_vc_counts.values,
                    color=BLUE_PALETTE[:len(infra_vc_counts)], edgecolor='white')
axes[0].set_yticks(range(len(infra_vc_counts)-1,-1,-1))
axes[0].set_yticklabels(infra_vc_counts.index, fontsize=9)
axes[0].set_xlabel('Number of Infrastructure Deals')
axes[0].set_title('Top 20 VCs in Infrastructure', pad=10)
for v, bar in zip(infra_vc_counts.values, bars):
    axes[0].text(v+0.1, bar.get_y()+bar.get_height()/2, str(v), va='center', fontsize=9)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

infra_stage = infra_deals['Stage'].value_counts()
axes[1].pie(infra_stage.values, labels=infra_stage.index, autopct='%1.1f%%',
            colors=BLUE_PALETTE[:len(infra_stage)],
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Infrastructure Deals by Stage', pad=10)
plt.tight_layout()
plt.savefig('../charts/06_infrastructure_deep_dive.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Total infrastructure deals: {len(infra_deals)}")
print(f"Total infrastructure capital: ${infra_deals['Fundraise'].sum():.0f}M")
print(f"Median infra deal size: ${infra_deals['Fundraise'].median():.0f}M")


### 5.2 Infrastructure Co-Investment Pairs

In [ ]:
infra_coinvest = Counter()
for project, group in infra_df.groupby('Project'):
    investors = group['Investors'].unique().tolist()
    if len(investors) >= 2:
        for pair in combinations(sorted(investors), 2):
            infra_coinvest[pair] += 1

top_infra_pairs = infra_coinvest.most_common(15)
pair_labels = [f"{p[0][:18]} + {p[1][:18]}" for p, c in top_infra_pairs]
pair_counts = [c for p, c in top_infra_pairs]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(pair_labels)-1,-1,-1), pair_counts,
               color=BLUE_PALETTE[:len(pair_labels)], edgecolor='white')
ax.set_yticks(range(len(pair_labels)-1,-1,-1))
ax.set_yticklabels(pair_labels, fontsize=9)
ax.set_xlabel('Number of Co-Investments')
ax.set_title('Top Co-Investment Pairs in Infrastructure', pad=15)
for bar, v in zip(bars, pair_counts):
    ax.text(v+0.05, bar.get_y()+bar.get_height()/2, str(v), va='center', fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/12_infra_coinvestment_pairs.png', dpi=200, bbox_inches='tight')
plt.show()

print("Top infrastructure co-investment pairs:")
for pair, count in top_infra_pairs[:10]:
    print(f"  {pair[0]} + {pair[1]}: {count} co-investments")


### 5.3 Syndicate Size Analysis

In [ ]:
syndicate_sizes = df.groupby('Project')['Investors'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].hist(syndicate_sizes, bins=range(1, syndicate_sizes.max()+2), color=COLORS['accent'],
             edgecolor='white', alpha=0.9, align='left')
axes[0].axvline(syndicate_sizes.median(), color=COLORS['highlight'], linestyle='--', linewidth=2,
                label=f'Median: {syndicate_sizes.median():.0f}')
axes[0].set_xlabel('Number of Investors per Deal'); axes[0].set_ylabel('Number of Deals')
axes[0].set_title('Syndicate Size Distribution'); axes[0].legend()
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

top_6_cats = unique_deals['Category'].value_counts().head(6).index
synd_by_cat = df.groupby(['Project','Category'])['Investors'].nunique().reset_index()
synd_by_cat = synd_by_cat[synd_by_cat['Category'].isin(top_6_cats)]
cat_medians = synd_by_cat.groupby('Category')['Investors'].median().sort_values()

bars = axes[1].barh(range(len(cat_medians)), cat_medians.values, color=BLUE_PALETTE[:len(cat_medians)], edgecolor='white')
axes[1].set_yticks(range(len(cat_medians)))
axes[1].set_yticklabels(cat_medians.index, fontsize=10)
axes[1].set_xlabel('Median Syndicate Size'); axes[1].set_title('Median Syndicate Size by Category')
for bar, v in zip(bars, cat_medians.values):
    axes[1].text(v+0.1, bar.get_y()+bar.get_height()/2, f'{v:.1f}', va='center', fontsize=10)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/13_syndicate_analysis.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Key Findings & Recommendations

### For Infrastructure Startups Approaching Series A

**1. Target the Right Investors:** Robot Ventures, Coinbase Ventures, and Animoca Brands are the most active in infrastructure. Focus outreach on investors with a demonstrated infrastructure thesis.

**2. Leverage Syndicate Patterns:** Co-investment pairs that frequently invest together (e.g., Robot Ventures + Maven 11) should be approached as a unit. Securing one partner significantly increases the chance of the other joining.

**3. Prepare for Stage Transition:** Most infrastructure deals are at Seed stage. Series A candidates need to demonstrate clear traction metrics (active users, TVL, developer adoption, partnerships) to differentiate.

**4. Benchmark Valuations:** Infrastructure commands a ~10x median valuation multiple (pre-valuation / fundraise). Prepare valuation justification benchmarked against comparable recent rounds.

**5. Market Timing:** Monthly capital deployment varies significantly. Monitor deal flow patterns to time your raise during periods of higher market activity.

### Market Overview

- The crypto VC market deployed $6.9B across 338 deals in 6 months
- Blockchain Service, DeFi, and Infrastructure are the top 3 categories by deal count
- Seed rounds dominate by count, while later stages capture more capital per deal
- The VC landscape is highly fragmented: 1,298 unique investors, but top 256 account for 50% of activity
